# 02 - Lakehouse
Camadas Bronze, Silver e Gold do pipeline de dados.

## Setup

In [ ]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

!pip install pyspark==3.5.0 delta-spark==3.2.0 mlflow==2.10.0 -q

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("FraudDetection-Lakehouse")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

BASE_PATH   = "/content/lakehouse"
BRONZE_PATH = f"{BASE_PATH}/bronze/transactions_raw"
SILVER_PATH = f"{BASE_PATH}/silver/transactions_processed"
GOLD_PATH   = f"{BASE_PATH}/gold/ml_dataset"
PIPELINE_PATH = f"{BASE_PATH}/pipeline"

print(f"Spark {spark.version} pronto!")

## Bronze Layer
Dados brutos sem transformacao. Schema explicito para evitar inferencia incorreta.

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, LongType, IntegerType
)

schema = StructType([
    StructField("trans_date_trans_time", StringType(),  True),
    StructField("cc_num",               StringType(),  True),
    StructField("merchant",             StringType(),  True),
    StructField("category",             StringType(),  True),
    StructField("amt",                  DoubleType(),  True),
    StructField("first",                StringType(),  True),
    StructField("last",                 StringType(),  True),
    StructField("gender",               StringType(),  True),
    StructField("street",               StringType(),  True),
    StructField("city",                 StringType(),  True),
    StructField("state",                StringType(),  True),
    StructField("zip",                  StringType(),  True),
    StructField("lat",                  DoubleType(),  True),
    StructField("long",                 DoubleType(),  True),
    StructField("city_pop",             LongType(),    True),
    StructField("job",                  StringType(),  True),
    StructField("dob",                  StringType(),  True),
    StructField("trans_num",            StringType(),  True),
    StructField("unix_time",            LongType(),    True),
    StructField("merch_lat",            DoubleType(),  True),
    StructField("merch_long",           DoubleType(),  True),
    StructField("is_fraud",             IntegerType(), True),
])

df_bronze = (
    spark.read
    .option("header", "true")
    .schema(schema)
    .csv("/content/data/fraudTrain.csv")
)

df_bronze = df_bronze.drop("Unnamed: 0")

print(f"Registros carregados: {df_bronze.count():,}")
df_bronze.printSchema()

In [ ]:
df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(BRONZE_PATH)

print(f"Bronze salva: {BRONZE_PATH}")

## Silver Layer
Limpeza, feature engineering e agregacoes temporais.

In [ ]:
from pyspark.sql.functions import (
    to_timestamp, to_date, hour, dayofweek,
    col, floor, datediff, current_date,
    radians, sin, cos, atan2, sqrt, lit,
    lag, unix_timestamp, count, sum
)
from pyspark.sql.window import Window

df = spark.read.format("delta").load(BRONZE_PATH)

# Timestamp e features de tempo
df = df.withColumn("transaction_timestamp", to_timestamp("trans_date_trans_time"))
df = (df
    .withColumn("hour", hour("transaction_timestamp"))
    .withColumn("day_of_week", dayofweek("transaction_timestamp"))
    .withColumn("is_weekend", col("day_of_week").isin([1, 7]))
    .withColumn("is_night", (col("hour") >= 0) & (col("hour") <= 5))
)

# Idade do cliente
df = (df
    .withColumn("birth_date", to_date("dob"))
    .withColumn("customer_age", floor(datediff(current_date(), col("birth_date")) / 365))
)

# Distancia geografica (Haversine)
R = 6371.0
df = (df
    .withColumn("lat1", radians(col("lat")))
    .withColumn("lon1", radians(col("long")))
    .withColumn("lat2", radians(col("merch_lat")))
    .withColumn("lon2", radians(col("merch_long")))
    .withColumn("dlat", col("lat2") - col("lat1"))
    .withColumn("dlon", col("lon2") - col("lon1"))
    .withColumn("a",
        sin(col("dlat") / 2) ** 2 +
        cos(col("lat1")) * cos(col("lat2")) * sin(col("dlon") / 2) ** 2
    )
    .withColumn("distance_km", 2 * lit(R) * atan2(sqrt(col("a")), sqrt(1 - col("a"))))
)

# Window functions
window_card = Window.partitionBy("cc_num").orderBy("transaction_timestamp")
window_24h  = (
    Window.partitionBy("cc_num")
    .orderBy(col("unix_time").cast("long"))
    .rangeBetween(-86400, 0)
)

df = df.withColumn("previous_transaction_time", lag("transaction_timestamp").over(window_card))
df = df.withColumn(
    "seconds_since_last_transaction",
    unix_timestamp("transaction_timestamp") - unix_timestamp("previous_transaction_time")
)
df = df.withColumn("transactions_last_24h", count("*").over(window_24h))
df = df.withColumn("amount_last_24h", sum("amt").over(window_24h))

# Remove colunas intermediarias do Haversine
df = df.drop("lat1", "lon1", "lat2", "lon2", "dlat", "dlon", "a")

df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(SILVER_PATH)
print(f"Silver salva: {SILVER_PATH}")

## Gold Layer
Features prontas para ML. Pipeline salvo separadamente para reuso no streaming.

In [ ]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

df = spark.read.format("delta").load(SILVER_PATH)
df = df.fillna({"seconds_since_last_transaction": 999999})

numeric_features     = ["amt", "city_pop", "lat", "long", "merch_lat", "merch_long",
                         "hour", "day_of_week", "customer_age", "distance_km",
                         "seconds_since_last_transaction", "transactions_last_24h", "amount_last_24h"]
boolean_features     = ["is_weekend", "is_night"]
categorical_features = ["category", "gender", "state"]

indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_index", handleInvalid="keep") for c in categorical_features]
encoders = [OneHotEncoder(inputCol=f"{c}_index", outputCol=f"{c}_encoded") for c in categorical_features]

final_features = numeric_features + boolean_features + [f"{c}_encoded" for c in categorical_features]

assembler = VectorAssembler(inputCols=final_features, outputCol="features")

pipeline = Pipeline(stages=indexers + encoders + [assembler])
pipeline_model = pipeline.fit(df)

# Salva o pipeline para reusar no streaming
pipeline_model.save(PIPELINE_PATH)
print(f"Pipeline salvo: {PIPELINE_PATH}")

df_gold = pipeline_model.transform(df)
df_gold = df_gold.select("trans_num", "cc_num", "amt", "features", col("is_fraud").alias("label"))

df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
print(f"Gold salva: {GOLD_PATH}")
print(f"Registros: {df_gold.count():,}")